# dfstore — Feature Demo

This notebook walks through every feature of **dfstore**: save, get, list, info, versioning, search, soft/hard delete, and restore.

A temporary directory is used as the store so this notebook is self-contained and repeatable — it never touches `~/.dfstore`.

In [1]:
import shutil
import tempfile
from pathlib import Path

import pandas as pd
import polars as pl

import dfstore

---
## 1. Test DataFrames

Two DataFrames are used throughout this notebook:
- **employees** — a pandas DataFrame
- **products** — a polars DataFrame

In [10]:
employees = pd.DataFrame({
    "name":       ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "age":        [25, 30, 35, 28, 32],
    "department": ["Engineering", "Marketing", "Engineering", "HR", "Marketing"],
    "salary":     [90_000, 65_000, 110_000, 72_000, 68_000],
    "city":       ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"],
})

products = pl.DataFrame({
    "product_id":   [101, 102, 103, 104],
    "product_name": ["Widget A", "Widget B", "Gadget X", "Gadget Y"],
    "price":        [9.99, 14.99, 49.99, 79.99],
    "in_stock":     [True, True, False, True],
    "category":     ["Widget", "Widget", "Gadget", "Gadget"],
})

analysis_result = pd.DataFrame({
    "department": ["Engineering", "Marketing", "HR"],
    "avg_salary": [100_000, 66_500, 72_000],
    "num_employees": [2, 2, 1],
})

reference_data = pd.DataFrame({
    "reference_id": [1, 2, 3],
    "description": ["Reference A", "Reference B", "Reference C"],
})

print("employees (pandas):")
display(employees)
print("\nproducts (polars):")
display(products)
print("\nreference_data (pandas):")
display(reference_data)

employees (pandas):


,name,age,department,salary,city
0,Alice,25,Engineering,90000,New York
1,Bob,30,Marketing,65000,Los Angeles
2,Charlie,35,Engineering,110000,Chicago
3,Diana,28,HR,72000,Houston
4,Eve,32,Marketing,68000,Phoenix



products (polars):


product_id,product_name,price,in_stock,category
i64,str,f64,bool,str
101,"""Widget A""",9.99,true,"""Widget"""
102,"""Widget B""",14.99,true,"""Widget"""
103,"""Gadget X""",49.99,false,"""Gadget"""
104,"""Gadget Y""",79.99,true,"""Gadget"""



reference_data (pandas):


,reference_id,description
0,1,Reference A
1,2,Reference B
2,3,Reference C


---
## 2. `save` — Store a DataFrame

`dfstore.save(df, name, description, tags, notes)` saves the DataFrame as parquet and records metadata (shape, dtypes, null counts, statistics).

Tags can be plain strings or `{"key": "value"}` dicts.

In [3]:
# Save pandas DataFrame
vr = dfstore.save(
    employees,
    name="employees",
    description="Company employee roster",
    tags=["hr", "internal", {"env": "production"}],
    notes="Initial load from HR system",   # overridden to 'Initial save' for v1
    # store_path=DEMO_STORE,
)

print(f"Saved  : employees")
print(f"Version: {vr.version}")
print(f"Shape  : {vr.shape}")
print(f"Library: {vr.library}")
print(f"Notes  : {vr.notes}")          # always 'Initial save' for v1

Saved  : employees
Version: 2
Shape  : (5, 5)
Library: pandas
Notes  : Initial load from HR system


In [6]:
# Save polars DataFrame
vr2 = dfstore.save(
    products,
    name="products",
    description="Product catalog from ERP",
    tags=["inventory", {"env": "staging"}],
)

print(f"Saved  : products")
print(f"Version: {vr2.version}")
print(f"Shape  : {vr2.shape}")
print(f"Library: {vr2.library}")        # 'polars' — detected automatically

Saved  : products
Version: 2
Shape  : (4, 5)
Library: polars


In [11]:
vr3 = dfstore.save(
    analysis_result,
    name="analysis_result",
    description="Average salary and employee count by department",
    tags=["hr", "analysis"],
)

In [12]:
vr4 = dfstore.save(
    reference_data,
    name="reference_data",
    description="Reference data for analysis",
    tags=["reference", "analysis"],
)

---
## 3. `list` — See all stored DataFrames

In [9]:
dfstore.list()

,name,description,tags,created_at,updated_at,current_version,deleted
0,products,Product catalog from ERP,"[inventory, {'env': 'staging'}]",2026-04-05 10:56:13.851230+00:00,2026-04-05 10:56:16.735146+00:00,2,False
1,employees,Company employee roster,"[hr, internal, {'env': 'production'}]",2026-04-04 20:06:14.353976+00:00,2026-04-05 10:56:03.642530+00:00,2,False


In [8]:
records = dfstore.list(format="raw")

summary = pd.DataFrame([{
    "name":         r.name,
    "description":  r.description,
    "tags":         str(r.tags),
    "version":      r.current_version,
    "shape":        r.versions[-1].shape,
    "last_updated": r.updated_at.strftime("%Y-%m-%d %H:%M:%S"),
} for r in records])

display(summary)

,name,description,tags,version,shape,last_updated
0,products,Product catalog from ERP,"['inventory', {'env': 'staging'}]",2,"(4, 5)",2026-04-05 10:56:16
1,employees,Company employee roster,"['hr', 'internal', {'env': 'production'}]",2,"(5, 5)",2026-04-05 10:56:03


---
## 4. `info` — Full metadata for a single DataFrame

Returns a `DFRecord` with all fields including column dtypes, null counts, and describe statistics.

In [6]:
r = dfstore.info("employees", store_path=DEMO_STORE)
vr = r.versions[-1]

print(f"Name        : {r.name}")
print(f"Description : {r.description}")
print(f"Tags        : {r.tags}")
print(f"Created     : {r.created_at.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Updated     : {r.updated_at.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Version     : {r.current_version}")
print(f"Shape       : {vr.shape}")
print(f"Deleted     : {r.deleted}")
print()
print("Dtypes:")
for col, dtype in vr.dtypes.items():
    print(f"  {col:15s} {dtype}")
print()
print("Null counts:", vr.null_counts)
print()
print("Describe (numeric cols):")
display(pd.DataFrame(vr.describe))

Name        : employees
Description : Company employee roster
Tags        : ['hr', 'internal', {'env': 'production'}]
Created     : 2026-04-04 20:00:23
Updated     : 2026-04-04 20:00:23
Version     : 1
Shape       : (5, 5)
Deleted     : False

Dtypes:
  name            object
  age             int64
  department      object
  salary          int64
  city            object

Null counts: {'name': 0, 'age': 0, 'department': 0, 'salary': 0, 'city': 0}

Describe (numeric cols):


,age,salary
count,5.000000,5.000000
mean,30.000000,81000.000000
std,3.807887,18894.443628
min,25.000000,65000.000000
25%,28.000000,68000.000000
50%,30.000000,72000.000000
75%,32.000000,90000.000000
max,35.000000,110000.000000


---
## 5. `get` — Retrieve a DataFrame

By default returns the same library type the DataFrame was saved with. Use `as_library` to convert.

In [ ]:
# Get latest version — returns pandas (same as saved)
df = dfstore.get("employees", store_path=DEMO_STORE)
print(f"Type  : {type(df).__name__}")
print(f"Shape : {df.shape}")
display(df)

In [ ]:
# Get polars DataFrame — returns polars (same as saved)
pl_df = dfstore.get("products", store_path=DEMO_STORE)
print(f"Type  : {type(pl_df).__name__}")
print(f"Shape : {pl_df.shape}")
display(pl_df)

In [7]:
# Cross-library conversion: products was saved as polars, retrieve as pandas
pd_from_polars = dfstore.get("products", as_library="pandas", store_path=DEMO_STORE)
print(f"Type after as_library='pandas': {type(pd_from_polars).__name__}")
display(pd_from_polars)

Type after as_library='pandas': DataFrame


,product_id,product_name,price,in_stock,category
0,101,Widget A,9.99,True,Widget
1,102,Widget B,14.99,True,Widget
2,103,Gadget X,49.99,False,Gadget
3,104,Gadget Y,79.99,True,Gadget


---
## 6. Versioning — Re-saving updates the version

When you save a DataFrame under an existing name, dfstore increments the version, records shape/column diffs, and keeps all previous versions intact.

In [8]:
# Give everyone a 10% raise and add a 'seniority' column
employees_v2 = employees.copy()
employees_v2["salary"] = (employees_v2["salary"] * 1.10).astype(int)
employees_v2["seniority"] = [3, 7, 12, 2, 5]

vr_v2 = dfstore.save(
    employees_v2,
    name="employees",
    notes="Annual raise + seniority column added",
    store_path=DEMO_STORE,
)

print(f"New version    : {vr_v2.version}")
print(f"Shape          : {vr_v2.shape}")
print(f"shape_diff     : {vr_v2.shape_diff}    ← (row_delta, col_delta)")
print(f"columns_added  : {vr_v2.columns_added}")
print(f"columns_removed: {vr_v2.columns_removed}")
print(f"row_diff       : {vr_v2.row_diff}")

New version    : 2
Shape          : (5, 6)
shape_diff     : (0, 1)    ← (row_delta, col_delta)
columns_added  : ['seniority']
columns_removed: []
row_diff       : 0


---
## 7. `versions` — View version history

In [9]:
version_history = dfstore.versions("employees", store_path=DEMO_STORE)

history_df = pd.DataFrame([{
    "version":          v.version,
    "saved_at":         v.saved_at.strftime("%Y-%m-%d %H:%M:%S"),
    "notes":            v.notes,
    "shape":            v.shape,
    "shape_diff":       v.shape_diff,
    "columns_added":    v.columns_added,
    "columns_removed":  v.columns_removed,
    "row_diff":         v.row_diff,
} for v in version_history])

display(history_df)

,version,saved_at,notes,shape,shape_diff,columns_added,columns_removed,row_diff
0,1,2026-04-04 20:00:23,Initial save,"(5, 5)",None,[],[],0
1,2,2026-04-04 20:02:19,Annual raise + seniority column added,"(5, 6)","(0, 1)",[seniority],[],0


---
## 8. `get` a specific version

Retrieve any historical version by passing `version=N`.

In [10]:
df_v1 = dfstore.get("employees", version=1, store_path=DEMO_STORE)
df_v2 = dfstore.get("employees", version=2, store_path=DEMO_STORE)

print("v1 columns:", list(df_v1.columns))
print("v2 columns:", list(df_v2.columns))
print()
print(f"v1 salary sum: {df_v1['salary'].sum():,}")
print(f"v2 salary sum: {df_v2['salary'].sum():,}  ← 10% higher")
print()
print("v1:")
display(df_v1)
print("v2:")
display(df_v2)

v1 columns: ['name', 'age', 'department', 'salary', 'city']
v2 columns: ['name', 'age', 'department', 'salary', 'city', 'seniority']

v1 salary sum: 405,000
v2 salary sum: 445,500  ← 10% higher

v1:


,name,age,department,salary,city
0,Alice,25,Engineering,90000,New York
1,Bob,30,Marketing,65000,Los Angeles
2,Charlie,35,Engineering,110000,Chicago
3,Diana,28,HR,72000,Houston
4,Eve,32,Marketing,68000,Phoenix


v2:


,name,age,department,salary,city,seniority
0,Alice,25,Engineering,99000,New York,3
1,Bob,30,Marketing,71500,Los Angeles,7
2,Charlie,35,Engineering,121000,Chicago,12
3,Diana,28,HR,79200,Houston,2
4,Eve,32,Marketing,74800,Phoenix,5


---
## 9. `search` — Find DataFrames by description, tags, or columns

All criteria are ANDed together. At least one must be provided.

In [ ]:
# Search by description (case-insensitive substring)
results = dfstore.search(description="catalog", store_path=DEMO_STORE)
print(f"search(description='catalog') → {[r.name for r in results]}")

In [ ]:
# Search by plain string tag
results = dfstore.search(tags=["hr"], store_path=DEMO_STORE)
print(f"search(tags=['hr'])                  → {[r.name for r in results]}")

# Search by dict tag
results = dfstore.search(tags=[{"env": "production"}], store_path=DEMO_STORE)
print(f"search(tags=[{{'env':'production'}}])   → {[r.name for r in results]}")

results = dfstore.search(tags=[{"env": "staging"}], store_path=DEMO_STORE)
print(f"search(tags=[{{'env':'staging'}}])      → {[r.name for r in results]}")

In [ ]:
# Search by column names — ALL listed columns must be present
results = dfstore.search(columns=["seniority"], store_path=DEMO_STORE)
print(f"search(columns=['seniority'])        → {[r.name for r in results]}")

results = dfstore.search(columns=["price", "in_stock"], store_path=DEMO_STORE)
print(f"search(columns=['price','in_stock']) → {[r.name for r in results]}")

results = dfstore.search(columns=["nonexistent"], store_path=DEMO_STORE)
print(f"search(columns=['nonexistent'])      → {[r.name for r in results]}")

In [ ]:
# Combined search — description AND tags (both must match)
results = dfstore.search(description="employee", tags=["hr"], store_path=DEMO_STORE)
print(f"search(description='employee', tags=['hr']) → {[r.name for r in results]}")

# No criteria raises ValueError
try:
    dfstore.search(store_path=DEMO_STORE)
except ValueError as e:
    print(f"\nNo criteria raises ValueError: {e}")

---
## 10. `delete` — Soft delete

Soft delete hides the DataFrame from `list()` and `search()`, and prevents `get()` from returning it. The data and metadata are preserved — the record is just flagged.

`info()` still works on soft-deleted items (useful for inspection).

In [ ]:
dfstore.delete("products", hard=False, store_path=DEMO_STORE)

active       = dfstore.list(store_path=DEMO_STORE)
with_deleted = dfstore.list(include_deleted=True, store_path=DEMO_STORE)

print(f"list()                     → {len(active)} record(s):       {[r.name for r in active]}")
print(f"list(include_deleted=True) → {len(with_deleted)} record(s): {[r.name for r in with_deleted]}")

# info() returns the record even when deleted
info_deleted = dfstore.info("products", store_path=DEMO_STORE)
print(f"\ninfo('products').deleted = {info_deleted.deleted}")

# get() raises DFNotFoundError
try:
    dfstore.get("products", store_path=DEMO_STORE)
except dfstore.DFNotFoundError as e:
    print(f"get('products') raises DFNotFoundError: {e}")

# search() excludes soft-deleted
search_result = dfstore.search(tags=["inventory"], store_path=DEMO_STORE)
print(f"search(tags=['inventory']) → {[r.name for r in search_result]}  (empty because deleted)")

---
## 11. `restore` — Undo a soft delete

In [ ]:
dfstore.restore("products", store_path=DEMO_STORE)

active = dfstore.list(store_path=DEMO_STORE)
print(f"After restore — active records: {[r.name for r in active]}")
print(f"info('products').deleted = {dfstore.info('products', store_path=DEMO_STORE).deleted}")

# Data is fully intact after restore
df_restored = dfstore.get("products", store_path=DEMO_STORE)
print(f"\nRestored products ({type(df_restored).__name__}):")
display(df_restored)

---
## 12. `delete(hard=True)` — Hard delete

Removes the record from `index.json` **and** deletes all parquet files. Irreversible.

In [ ]:
data_dir = DEMO_STORE / "data" / "products"
print(f"data/products/ exists before hard delete: {data_dir.exists()}")

dfstore.delete("products", hard=True, store_path=DEMO_STORE)

print(f"data/products/ exists after hard delete:  {data_dir.exists()}")

all_records = dfstore.list(include_deleted=True, store_path=DEMO_STORE)
print(f"Total records (including deleted): {len(all_records)} → {[r.name for r in all_records]}")

# info() now raises
try:
    dfstore.info("products", store_path=DEMO_STORE)
except dfstore.DFNotFoundError as e:
    print(f"info('products') raises DFNotFoundError: {e}")

---
## 13. Error cases

dfstore raises typed exceptions for all error conditions.

In [ ]:
# Invalid name (spaces, special chars not in [a-zA-Z0-9_-])
try:
    dfstore.save(employees, name="my df!", store_path=DEMO_STORE)
except ValueError as e:
    print(f"ValueError  (invalid name)   : {e}")

# Non-DataFrame input
try:
    dfstore.save({"not": "a dataframe"}, name="bad", store_path=DEMO_STORE)
except TypeError as e:
    print(f"TypeError   (wrong type)     : {e}")

# Get non-existent name
try:
    dfstore.get("nonexistent", store_path=DEMO_STORE)
except dfstore.DFNotFoundError as e:
    print(f"DFNotFoundError (not found)  : {e}")

# Get out-of-range version
try:
    dfstore.get("employees", version=99, store_path=DEMO_STORE)
except ValueError as e:
    print(f"ValueError  (bad version)    : {e}")

# Restore something that isn't deleted
try:
    dfstore.restore("employees", store_path=DEMO_STORE)
except dfstore.DFStoreError as e:
    print(f"DFStoreError (not deleted)   : {e}")

# Save to a soft-deleted name
dfstore.delete("employees", store_path=DEMO_STORE)
try:
    dfstore.save(employees, name="employees", store_path=DEMO_STORE)
except dfstore.DFStoreError as e:
    print(f"DFStoreError (soft-deleted)  : {e}")
dfstore.restore("employees", store_path=DEMO_STORE)  # clean up

---
## 14. Storage layout

Inspect what dfstore writes to disk.

In [ ]:
import os

print(f"Store root: {DEMO_STORE}\n")
for root, dirs, files in os.walk(DEMO_STORE):
    level = root.replace(str(DEMO_STORE), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    for f in files:
        fpath = Path(root) / f
        size = fpath.stat().st_size
        print(f"{indent}  {f}  ({size:,} bytes)")

In [ ]:
import json

with open(DEMO_STORE / "index.json") as f:
    index = json.load(f)

# Pretty-print the employees record (truncate describe for readability)
emp = json.loads(json.dumps(index["employees"]))
for v in emp["versions"]:
    v["describe"] = {col: "..." for col in v["describe"]}

print(json.dumps(emp, indent=2))

---
## 15. CLI Demo

The same operations are available from the shell via the `dfstore` command.

In [ ]:
import subprocess, sys

def cli(*args):
    """Run a dfstore CLI command against DEMO_STORE and print output."""
    env = {**os.environ, "DFSTORE_PATH": str(DEMO_STORE)}
    result = subprocess.run(
        [sys.executable, "-m", "dfstore.cli", *args],
        capture_output=True, text=True, env=env
    )
    output = result.stdout + result.stderr
    print(f"$ dfstore {' '.join(args)}")
    print(output.rstrip())
    print()

cli("list")

In [ ]:
cli("info", "employees")

In [ ]:
cli("versions", "employees")

In [ ]:
cli("get", "employees", "--format", "csv")

In [ ]:
cli("search", "--description", "employee")
cli("search", "--tags", "hr")
cli("search", "--columns", "seniority")

In [ ]:
# Save a CSV file via CLI
import tempfile
csv_path = Path(tempfile.mktemp(suffix=".csv"))
employees.to_csv(csv_path, index=False)

cli("save", str(csv_path), "--name", "employees_csv", "--description", "Saved from CSV",
    "--tags", "hr", "--tags", "env=test")
cli("list")

csv_path.unlink(missing_ok=True)

---
## 16. Cleanup

Remove the temporary store directory.

In [ ]:
shutil.rmtree(DEMO_STORE)
print(f"Store removed: {DEMO_STORE}")
print("Re-run the setup cell (cell 2) to start a fresh session.")